In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from search.parallel_gradient import ParallelGradientDescent
from utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#look for experiment files in parents
import os
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)


In [ ]:
experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")

In [ ]:
dataset ="bigger_mnist"
default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}
architecture = default_architecutre_mapping[dataset]
budget = 120

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data)


In [ ]:
dataset_info

In [ ]:
dataset_dict.keys()

In [ ]:
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']



In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]

In [ ]:
#test images
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 5))

axs[0].imshow(torchvision.utils.make_grid(next(iter(train_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(next(iter(val_loader))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[2].imshow(torchvision.utils.make_grid(next(iter(test_loader_transformed))[0], nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Training')
axs[1].set_title('Validation')
axs[2].set_title('Test')

for ax in axs.flat:
    ax.axis('off')

In [ ]:
from experiment_thesis.main import train_and_get_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

In [ ]:
model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,"different_domain")
os.makedirs(results_dir_path, exist_ok=True)
def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, f"{safe}.json")


In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": 100,
        "precision": "16-mixed",
},load_if_exists=True)


res = evaluate_base_model(model, test_loader_transformed, device)
print(res)

In [ ]:
dataset_info

In [ ]:
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images
if dataset_info.transform_seq_name is not None and dataset_info.datatype == "image":
    transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    )

In [ ]:
transform_seq.cuda()

In [ ]:
print(list(model.named_modules()))

In [ ]:
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache
transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model,train_loader_no_shuffle,cache_dir=os.path.join(embedding_cache_path,cache_name_train))
embeddings_t,final_t,classes_t = cache_train("fc",capture_modes='input',flatten=True)

In [ ]:
dual_output_model = cache_train.make_wrapper("fc",capture_modes='input',concat=False,flatten=True)


In [ ]:
inter,final = dual_output_model(torch.randn(2,1,28,28).to(device))

In [ ]:
from utils.eval.ood_performance import evaluate_confidence_module
# Add cached runner
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
import json, math, pandas as pd



In [ ]:
from utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence,PerClassKNNConfidence

from confidence.input_transform import InputTransformImage,PCAInputModule,RandomProjectionModule
input_transform_image = InputTransformImage((3,3),(128,7,7))
input_transform_pca = PCAInputModule(512)
input_transform_random = RandomProjectionModule(512,method="gaussian")


nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine",input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t,classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained,EnergyConfidence(), mult=False,b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model,conf_split_pretrained,index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq,consolidate_method="consolidate_simple")


In [ ]:
from search import shgo

random_search = shgo.SHGO(initial_samples=120, local_runs=1, local_max_steps=0,project_param=False)
model.cuda().eval()

In [ ]:
from utils.transform_sequence import TransformSequence
from utils.affine_transforms import AffineTransformation2D

transformation_direct =[AffineTransformation2D.DIRECT.value]
transform_seq_direct = TransformSequence(
    transformations=transformation_direct,
    domains=[((-1.2,1.2),(-1.2,1.2),(-1.2,1.2),(-1.2,1.2),(-0.01,0.01),(-0.01,0.01),)],
    device=device,
    dtype=torch.float32,
    reflect=False,
)

In [ ]:
repeats =4
overwrite = False

In [ ]:


load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_default")
)

In [ ]:
problem_nn_pytorch_pretrained_direct = TransformationProblem(conf_mod_nn_pytorch_pretrained,transform_seq_direct,consolidate_method="consolidate_simple")
ramdom_search_huge= shgo.SHGO(initial_samples=120, local_runs=1, local_max_steps=0,project_param=False)



load_or_run_evaluate_confidence_and_search(
    model, optimizer=ramdom_search_huge, problem=problem_nn_pytorch_pretrained_direct,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_matrix")
)

In [ ]:
import torch
import matplotlib.pyplot as plt
from utils.eval.ood_performance import ITSWRAPPER

def prepare_img_for_plot(img):
    """
    Convert tensor to numpy array suitable for imshow.
    Clips/normalizes values to [0,1].
    """
    img = img.detach().cpu()

    if img.dim() == 3 and img.shape[0] in [1, 3]:
        img = img.permute(1, 2, 0)  # C,H,W -> H,W,C

    img = img.numpy()

    # Normalize if not in [0,1]
    min_val, max_val = img.min(), img.max()
    #clip
    img = np.clip(img, 0, 1)
    return img


def plot_transformed_batch(data_loader, problem, optimizer, k=8, device=None):
    """
    Fetches the first batch, applies transformation, and plots the first k samples.
    """
    with torch.no_grad():
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        data_iter = iter(data_loader)
        data, _ = next(data_iter)
        data = data[:k].to(device)

        # Optimize / transform
        if not isinstance(optimizer, ITSWRAPPER):
            res = optimizer.optimize(problem, data, y=None)
            print(res[1].detach().cpu().numpy()[:k])
            x_trans = problem.transform(data, res[0])
            conf = problem.confidence_module(x_trans, None)[0]
            print(conf.detach().cpu().numpy()[:k])
        else:
            x_trans, _ = optimizer.optimize(problem, data)

        x_trans = x_trans.detach().cpu()
        data = data.detach().cpu()

        # --- Plot original ---
        plt.figure(figsize=(k * 2, 2))
        for i in range(min(k, data.shape[0])):
            plt.subplot(1, k, i + 1)
            img = prepare_img_for_plot(data[i])
            if img.ndim == 2 or img.shape[-1] == 1:
                plt.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
            else:
                plt.imshow(img)
            plt.axis('off')
        plt.tight_layout()
        plt.show()

        # --- Plot transformed ---
        plt.figure(figsize=(k * 2, 2))
        for i in range(min(k, x_trans.shape[0])):
            plt.subplot(1, k, i + 1)
            img = prepare_img_for_plot(x_trans[i])
            if img.ndim == 2 or img.shape[-1] == 1:
                plt.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
            else:
                plt.imshow(img)
            plt.axis('off')
        plt.tight_layout()
        plt.show()


In [ ]:
model.eval()

In [ ]:
problem_nn_pytorch_pretrained_direct.max_batch_size=1280

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
plot_transformed_batch(test_loader_transformed, problem_nn_pytorch_pretrained_direct, random_search, k=8, device=device)

In [ ]:
problem_nn_pytorch_pretrained.max_batch_size =1280

In [ ]:
plot_transformed_batch(test_loader_transformed, problem_nn_pytorch_pretrained, random_search, k=8, device=device)


In [ ]:
import torch

# 1) Replace angle rotation with complex rotation; keep shears and scales unchanged
transformations_rot_complex = [
    AffineTransformation2D.ROTATION_COMPLEX.value,
    AffineTransformation2D.SHEARING_X.value,
    AffineTransformation2D.SHEARING_Y.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
]
domains_rot_complex = [
    (-torch.pi, torch.pi),                    # complex rotation angle-like param
    ((-0.5, 0.5),),                           # shear x
    ((-0.5, 0.5),),                           # shear y
    ((1 / 1.8 - 1, 0.8),),                    # scale x delta
    ((1 / 1.8 - 1, 0.8),),                    # scale y delta
]
transform_seq_rot_complex = TransformSequence(
    transformations=transformations_rot_complex,
    domains=domains_rot_complex,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_rot_complex = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_rot_complex,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_rot_complex, random_search, k=8, device=device)


# Complex rotation (cached)
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_rot_complex,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_rot_complex")
)

In [ ]:



# 2) Replace angle rotation with skew rotation; keep shears and scales unchanged
transformations_rot_skew = [
    AffineTransformation2D.ROTATION_SKEW.value,
    AffineTransformation2D.SHEARING_X.value,
    AffineTransformation2D.SHEARING_Y.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
]
domains_rot_skew = [
    (-torch.pi, torch.pi),                    # skew rotation angle-like param
    ((-0.5, 0.5),),                           # shear x
    ((-0.5, 0.5),),                           # shear y
    ((1 / 1.8 - 1, 0.8),),                    # scale x delta
    ((1 / 1.8 - 1, 0.8),),                    # scale y delta
]
transform_seq_rot_skew = TransformSequence(
    transformations=transformations_rot_skew,
    domains=domains_rot_skew,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_rot_skew = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_rot_skew,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_rot_skew, random_search, k=8, device=device)


# Skew rotation (cached)
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_rot_skew,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_rot_skew")
)

In [ ]:

# 3) Replace shear x+y and scale x+y with a single SHEARSCALE; keep angle rotation
transformations_shearscale = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.SHEARSCALE.value,
]
domains_shearscale = [
    (-torch.pi, torch.pi),                    # angle rotation
    (
        (1 / 1.8 - 1, 0.8),                   # scale x delta
        (1 / 1.8 - 1, 0.8),                   # scale y delta
        (-0.5, 0.5),                          # shear x
    ),
]
transform_seq_shearscale = TransformSequence(
    transformations=transformations_shearscale,
    domains=domains_shearscale,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_shearscale = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_shearscale,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_shearscale, random_search, k=8, device=device)


# ShearScale (cached)
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_shearscale,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_shearscale")
)

In [ ]:
x =next(iter(test_loader))[0].to(device)[0].repeat(batch_size,1,1,1)
p = torch.zeros((x.shape[0],4)).cuda()
p[:,3]=torch.linspace(-0.5,0.5,x.shape[0])
xt =transform_seq_shearscale.transform(x,p)

#plot them
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10, 5))
axs[0].imshow(torchvision.utils.make_grid(x, nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[1].imshow(torchvision.utils.make_grid(xt, nrow=batch_size//4).permute(1, 2, 0).cpu())
axs[0].set_title('Original')
axs[1].set_title('Transformed')
for ax in axs.flat:
    ax.axis('off')


In [ ]:

# 4) Add translation together with SHEARSCALE; keep angle rotation
transformations_trans_shearscale = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.SHEARING.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
    #AffineTransformation2D.TRANSLATION.value,
]
domains_trans_shearscale = [
            (-torch.pi, torch.pi),
            ((-0.5, 0.5),),
            ((1/(1+0.8)-1, 0.8),),
            ((1/(1+0.8)-1, 0.8),)
        ]
transform_seq_trans_shearscale = TransformSequence(
    transformations=transformations_trans_shearscale,
    domains=domains_trans_shearscale,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_trans_shearscale = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_trans_shearscale,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_trans_shearscale, random_search, k=8, device=device)


load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_trans_shearscale,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_shearxy")
)



In [ ]:

# 4) Add translation together with SHEARSCALE; keep angle rotation
transformations_trans_shearscale = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.SHEARING.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
    AffineTransformation2D.TRANSLATION.value,
]
domains_trans_shearscale = [
    (-torch.pi, torch.pi),                    # angle rotation
    ((-0.5, 0.5),),                           # shear x
    ((1 / 1.8 - 1, 0.8),),                    # scale x delta
    ((1 / 1.8 - 1, 0.8),),                    # scale y delta
    ((-0.1, 0.1), (-0.1, 0.1)),           # translation x,y
]
transform_seq_trans_shearscale = TransformSequence(
    transformations=transformations_trans_shearscale,
    domains=domains_trans_shearscale,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_trans_shearscale = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_trans_shearscale,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_trans_shearscale, random_search, k=8, device=device)

load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search,
    problem=problem_trans_shearscale,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_translation"))


In [ ]:

# 4) Add translation together with SHEARSCALE; keep angle rotation
transformations_trans_shearscale = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.SHEARING_X.value,
    AffineTransformation2D.SCALING_X.value,
    AffineTransformation2D.SCALING_Y.value,
]
domains_trans_shearscale = [
            (-torch.pi, torch.pi),
            ((-0.5, 0.5),),
            ((1/(1+0.8)-1, 0.8),),
            ((1/(1+0.8)-1, 0.8),)
        ]
transform_seq_trans_shearscale = TransformSequence(
    transformations=transformations_trans_shearscale,
    domains=domains_trans_shearscale,
    device=device,
    dtype=torch.float32,
    reflect=False,
)
problem_trans_shearscale = TransformationProblem(
    conf_mod_nn_pytorch_pretrained, transform_seq_trans_shearscale,
    consolidate_method="consolidate_simple", max_batch_size=1280
)
plot_transformed_batch(test_loader_transformed, problem_trans_shearscale, random_search, k=8, device=device)


load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_trans_shearscale,
    test_loader=test_loader_transformed, max_batch_override=1280,
    repeats=repeats, overwrite=overwrite, save_path=savepath("random_search_knn_shear_x")
)



In [ ]:
#ow test different ways to scale the image
from utils.transforms.apply import grid_resample,grid_resample_bicubic,grid_resample_nearest,grid_resample_blur_simple,grid_resample_border,grid_resample_blur,grid_resample_reflection,grid_resample_deblur

In [ ]:
import os
from utils.eval.vis import process_and_plot_groups, plt_setup_latex

json_files = sorted([f for f in os.listdir(results_dir_path) if f.endswith(".json")])
if not json_files:
    print(f"No .json result files found in `{results_dir_path}`")
else:
    items = [(os.path.splitext(f)[0], f) for f in json_files]
    groups = [("all_results", "All results", items)]
    process_and_plot_groups(groups, results_dir_path, architecture)

In [ ]:
json_files

In [ ]:
import os
from utils.eval.vis import process_and_plot_groups

# `results_dir_path` and `architecture` are already defined in the notebook;
# `results_dir_path` should point to the directory containing the .json result files
json_files = sorted([f for f in os.listdir(results_dir_path) if f.endswith(".json")])
#remove matrix one random_search_knn_matrix.json
json_files = [f for f in json_files if f != "random_search_knn_matrix.json"]

if not json_files:
    print(f"No .json result files found in `{results_dir_path}`")
else:
    items = [(os.path.splitext(f)[0], f) for f in json_files]
    groups = [("all_results", "All results", items)]
    process_and_plot_groups(groups, results_dir_path, architecture)

In [ ]:
# python
# Ensure inline backend (run once at top of notebook)
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")

# Call processing but do not save PGF (this will call plt.show() inside plot_group_short)
from utils.eval.vis import process_and_plot_groups

json_files = sorted([f for f in os.listdir(results_dir_path) if f.endswith(".json")])
if not json_files:
    print(f"No .json result files found in `{results_dir_path}`")
else:
    items = [(os.path.splitext(f)[0], f) for f in json_files]
    groups = [("all_results", "All results", items)]
    process_and_plot_groups(groups, results_dir_path, architecture, save_pgf=False)

In [ ]:
W = plt_setup_latex()

#set figure zie to W,W*0.6
plt.rcParams['figure.figsize'] = (W, W * 0.6)

In [ ]:
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "a_different_domain", dataset,
                           transform_name)

In [ ]:
# python
# Build readable labels for result files and call the existing plotting helper.
import os
from utils.eval.vis import process_and_plot_groups

# directory containing the .json result files
# `results_dir_path` is expected to be defined earlier in the notebook
json_files = sorted([f for f in os.listdir(results_dir_path) if f.endswith(".json")])

# Optional: exclude the matrix result if present
json_files = [f for f in json_files if f != "random_search_knn_matrix.json"]

# explicit pretty names for known experiment files
pretty_names = {
    "random_search_knn_default": "Shear in x and y",
    "random_search_knn_shearscale": "Decomposed Shear",
    "random_search_knn_shearxy": "Unit Diagonal Shear",
    "random_search_knn_translation": "Full Shear + Translation",
    "random_search_knn_shear_x": "Shear in x only",
    "random_search_knn_matrix": "Random search — KNN (matrix)",
    # add more mappings as needed
}

def prettify_basename(basename: str) -> str:
    # fallback: convert snake_case to Title Case
    parts = basename.split("_")
    return " ".join(p.capitalize() for p in parts)

# items: list of tuples (label, filename) used by process_and_plot_groups
items = []
for fname in json_files:
    base = os.path.splitext(fname)[0]
    label = pretty_names.get(base, prettify_basename(base))
    #skip if not in pretty names
    if base not in pretty_names:
        continue
    items.append((label, fname))

if not items:
    print(f"No .json result files found in `{results_dir_path}`")
else:
    groups = [("all_results", "All results", items)]
    # call the plotting helper; keep `architecture` variable as before
    process_and_plot_groups(groups, results_dir_path, architecture, save_pgf=True,save_root=figure_path)
    process_and_plot_groups(groups, results_dir_path, architecture, save_pgf=False,save_root=figure_path)

In [ ]:
transform_seq_normal = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
    ).cuda()

In [ ]:
transform_seq_normal.domains

In [ ]:
#todo maybe test in another file what settings make sense

In [ ]:
transform_seq_normal.init_method = "individual"
transform_seq_normal.init_method = "latin_hypercube"